# AE3: Transformación y enriquecimiento del dataset Goodreads Books

Segunda etapa del proyecto integrador. Este notebook parte del mismo dataset de AE2 (`pfaha/goodreads-books`, revisión fijada), lo transforma y enriquece, responde las preguntas de negocio con una tabla por pregunta y exporta el dataset preparado, las tablas de resultados (`.csv`) y el resumen descriptivo (`.txt`) en `AE3/resultados/`.

**Necesidad (escenario hipotético, AE2).** Una librería quiere repartir su espacio entre *Lecturas Ágiles* y un área de obras extensas, y priorizar editoriales e idiomas en sus compras importadas. Necesita un diagnóstico descriptivo de los libros destacados de Goodreads (calificación promedio ≥ 3,5).

**Pregunta de análisis (AE2).** Entre los libros de Goodreads con calificación promedio igual o superior a 3,5, ¿cómo se distribuyen según su extensión en páginas (lecturas ágiles frente a obras extensas), qué editoriales e idiomas concentran más títulos en cada grupo y cómo varía la calificación promedio entre ambos grupos?

| # | Pregunta de negocio (objetivos específicos de AE2) | Tabla |
|---|---|---|
| P1 | ¿Cuál es la población de estudio y cuántos registros se excluyen por cada criterio? | `t1_control_poblacion.csv` |
| P2 | ¿Qué número y porcentaje de títulos reúne cada área con un umbral de páginas justificado? | `t2_reparto_por_extension.csv`, `t2b_sensibilidad_umbral.csv` |
| P3 | ¿Qué diez editoriales concentran más títulos destacados en cada área y con qué calificación promedio? | `t3_top_editoriales.csv` |
| P4 | ¿Qué diez idiomas concentran más títulos destacados en cada área y con qué calificación promedio? | `t4_top_idiomas.csv` |
| P5 | ¿Cómo se distribuye la calificación promedio en cada área? | `t5_calificacion_por_extension.csv` |
| P6 | ¿Cuál es la composición por idioma de cada área? | `t6_composicion_idioma.csv` |
| Complementaria | ¿Cuánta participación de lectores, pertenencia a series y qué formatos reúne cada área? | `t7_participacion_series.csv`, `t8_formato_por_extension.csv` |

**Ajustes respecto de AE2 (justificados).**
1. *Base de análisis.* En AE2 el documento describía el `df_transformado` de `actividad1.ipynb` (242.631 × 6), que eliminaba solo nulos y no depuraba páginas. AE3 parte del dataset completo con revisión fijada y aplica reglas explícitas de calidad de páginas (1 a 5.000), siguiendo la preparación acordada por el equipo en el notebook de AE2.
2. *Variables complementarias.* Se recuperan `ratings_count`, `reviews_count`, `is_part_of_series` y `format`, que el `df_transformado` había descartado. No cambian la pregunta principal; se usan en una tabla complementaria porque describen mejor cada área.
3. *Umbral de extensión.* Se fija en 350 páginas (cercano al percentil 75) en lugar de los cortes por cuartiles previstos en AE2; se acompaña de un análisis de sensibilidad.
4. *Nombre del área extensa.* Se usa "Obras extensas": el umbral se basa solo en páginas, así que el nombre "Grandes Clásicos y Sagas" prometía una distinción que los datos no hacen.

## 1. Configuración

In [1]:
import os
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Ruta dinámica hacia la raíz del repositorio (la carpeta que contiene 'src')
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from datasets import disable_progress_bars

import preparacion as prep
from hf_loader import HFDatasetLoader
from reportes import Reportes

disable_progress_bars()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Mismo dataset y revisión que AE2
DATASET_ID = "pfaha/goodreads-books"
DATASET_URL = f"https://huggingface.co/datasets/{DATASET_ID}"
CONFIG = "default"
SPLIT = "train"
REVISION = "43b3198cd7ec9c142d56181f1a09ed9e8c112610"

# Reglas de la preparación
CALIFICACION_MINIMA = 3.5
PAGINAS_MIN, PAGINAS_MAX = 1, 5000
UMBRAL_PAGINAS = 350
UMBRALES_CANDIDATOS = (250, 300, 350, 400)

RUTA_RESULTADOS = ROOT / "AE3" / "resultados"

print(f"Raíz del proyecto: {ROOT.name}")
print(f"Dataset: {DATASET_ID} | split: {SPLIT} | revisión: {REVISION[:7]}")

Raíz del proyecto: RepoMachine_ProInteg3
Dataset: pfaha/goodreads-books | split: train | revisión: 43b3198


## 2. Obtención del dataset de AE2

Se descarga el split `train` completo de la misma revisión usada en AE2 (no se usa una muestra). La descarga es pública y no requiere token; los datos quedan en la caché local de Hugging Face. `df_original` no se modifica: todas las transformaciones trabajan sobre copias.

In [2]:
loader = HFDatasetLoader(dataset_name=DATASET_ID, config=CONFIG, split=SPLIT, revision=REVISION)
df_original = loader.load()

assert df_original.shape == (500_000, 24), "La revisión fijada debe tener 500.000 filas y 24 columnas"
print(f"Dataset original: {df_original.shape[0]:,} filas | {df_original.shape[1]} columnas")

Dataset original: 500,000 filas | 24 columnas


## 3. Transformación y reglas de calidad

| Regla | Decisión | Motivo |
|---|---|---|
| Calificación ≥ 3,5 | Se conservan solo los libros destacados | Define la población de la necesidad |
| Calificación faltante | Se excluye | No permite saber si el libro es destacado |
| Páginas faltantes, ≤ 0 o > 5.000 | Se excluye del análisis por extensión y se cuenta aparte | No permiten asignar un área; valores como 0 o 640 millones son errores de captura |
| Editorial, idioma o formato faltante | Se conserva el registro y el faltante se reporta | Excluirlos sesgaría el reparto por extensión |
| Espacios externos y textos vacíos en editorial e idioma | Se quitan los espacios y los vacíos pasan a nulo | Evita categorías duplicadas o vacías |
| Ediciones de una misma obra | Se conservan, con `book_id` y `work_id` | Cada edición tiene su propia extensión; las métricas de lectores se tratan por obra |
| Valores originales | Se conservan `publisher_original` y `language_original` | Trazabilidad de la limpieza |

In [3]:
poblacion, t1_control_poblacion = prep.filtrar_poblacion(
    df_original, CALIFICACION_MINIMA, PAGINAS_MIN, PAGINAS_MAX
)
t1_control_poblacion

,criterio,registros,porcentaje_del_dataset
0,Dataset original,500000,100.00
1,Calificación faltante,258,0.05
2,Calificación < 3.5,223590,44.72
3,Calificación >= 3.5 (destacados),276152,55.23
4,Destacados con páginas faltantes,22129,4.43
5,Destacados con páginas < 1,18868,3.77
6,Destacados con páginas > 5.000,51,0.01
7,Población final: destacados con páginas entre ...,235104,47.02


In [4]:
columnas_fuente = [
    "book_id", "work_id", "title", "pages", "average_rating", "ratings_count",
    "reviews_count", "publisher", "language", "is_part_of_series", "series", "format",
]
df_final = poblacion[columnas_fuente].copy()

for campo in ["publisher", "language"]:
    df_final[f"{campo}_original"] = df_final[campo]
    df_final[campo] = prep.limpiar_categoria(df_final[campo])

calidad_poblacion = pd.DataFrame({
    "faltantes": [
        int(df_final["publisher"].isna().sum()),
        int(df_final["language"].isna().sum()),
        int(prep.agrupar_formato(df_final["format"]).eq("Sin informar").sum()),
        int(df_final["book_id"].duplicated().sum()),
        int(df_final["work_id"].isna().sum()),
    ],
}, index=[
    "Sin editorial (nulo o vacío)",
    "Sin idioma",
    "Sin formato (nulo, vacío o 'Unknown Binding')",
    "book_id duplicados",
    "work_id faltantes",
])
calidad_poblacion["porcentaje_poblacion"] = (calidad_poblacion["faltantes"] / len(df_final) * 100).round(2)

assert df_final["book_id"].is_unique
assert df_final["work_id"].notna().all()
calidad_poblacion

,faltantes,porcentaje_poblacion
Sin editorial (nulo o vacío),6915,2.94
Sin idioma,5576,2.37
"Sin formato (nulo, vacío o 'Unknown Binding')",3746,1.59
book_id duplicados,0,0.00
work_id faltantes,0,0.00


## 4. Enriquecimiento

Todas las variables nuevas se calculan con columnas del propio dataset; no se une ninguna tabla externa, por lo que no hay riesgo de multiplicar registros (se verifica con `assert`).

| Variable incorporada | Fórmula o regla | Significado | Utilidad |
|---|---|---|---|
| `categoria_extension` | `pages` ≤ 350 → "Lecturas Ágiles"; > 350 → "Obras extensas" | Área de exhibición de la edición | Variable central de P2 a P6 |
| `rango_calificacion` | `average_rating` en [3,5; 4,0), [4,0; 4,5), [4,5; 5,0] | Nivel de valoración | Distribución de la calificación (P5) |
| `idioma_normalizado` | Primer nombre de la etiqueta de idioma: "Spanish; Castilian" → "Spanish" | Idioma sin variantes de etiqueta | Ranking y composición por idioma (P4, P6) |
| `cantidad_series` | Número de series listadas en `series` | Cuántas sagas incluyen la edición | Complementa `is_part_of_series` |
| `formato_grupo` | Palabras clave sobre `format`: Tapa blanda, Tapa dura, Digital, Audio, Otro, Sin informar | Formato físico o digital | Describe cada área (complementaria) |
| `tasa_resenas` | `reviews_count / ratings_count` (nula si no hay calificaciones) | Proporción de calificaciones con reseña escrita | Intensidad de participación (complementaria) |
| `registro_representativo_obra` | Por `work_id`, la edición con más calificaciones (empate: menor `book_id`) | Un registro por obra | Sumar calificaciones y reseñas sin contar dos veces las ediciones |

In [5]:
filas_antes = len(df_final)

df_final["categoria_extension"] = prep.clasificar_extension(df_final["pages"], UMBRAL_PAGINAS)
df_final["rango_calificacion"] = prep.rango_calificacion(df_final["average_rating"])
df_final["idioma_normalizado"] = prep.normalizar_idioma(df_final["language_original"])
df_final["cantidad_series"] = df_final["series"].map(
    lambda valor: len(valor) if hasattr(valor, "__len__") and not isinstance(valor, str) else 0
)
df_final["formato_grupo"] = prep.agrupar_formato(df_final["format"])
df_final["tasa_resenas"] = prep.tasa_resenas(df_final)
df_final["registro_representativo_obra"] = prep.marcar_registro_representativo(df_final)

# Enteros sin decimales para que el CSV sea más legible y liviano
for campo in ["work_id", "pages", "ratings_count", "reviews_count"]:
    df_final[campo] = df_final[campo].astype("Int64")

df_final = df_final[[
    "book_id", "work_id", "title", "pages", "categoria_extension",
    "average_rating", "rango_calificacion", "ratings_count", "reviews_count", "tasa_resenas",
    "publisher_original", "publisher", "language_original", "idioma_normalizado",
    "is_part_of_series", "cantidad_series", "format", "formato_grupo",
    "registro_representativo_obra",
]]

assert len(df_final) == filas_antes, "El enriquecimiento no debe agregar ni quitar filas"
assert df_final["categoria_extension"].notna().all() and df_final["rango_calificacion"].notna().all()
assert df_final["registro_representativo_obra"].sum() == df_final["work_id"].nunique()
assert (df_final["is_part_of_series"] == df_final["cantidad_series"].gt(0)).mean() > 0.99

print(f"DataFrame final: {df_final.shape[0]:,} filas | {df_final.shape[1]} columnas")
df_final.head()

DataFrame final: 235,104 filas | 19 columnas


,book_id,work_id,title,pages,categoria_extension,average_rating,rango_calificacion,ratings_count,reviews_count,tasa_resenas,publisher_original,publisher,language_original,idioma_normalizado,is_part_of_series,cantidad_series,format,formato_grupo,registro_representativo_obra
0,7282453,44066165,La doctrina secreta. Vol I. Cosmogenesis (Cien...,319,Lecturas Ágiles,4.16,"4,0 a 4,49",255,22,0.0863,Editorial Kier,Editorial Kier,Spanish; Castilian,Spanish,True,1,Paperback,Tapa blanda,False
3,2027361,580945,Personality and Intellectual Competence,214,Lecturas Ágiles,3.88,"3,5 a 3,99",8,0,0.0000,Psychology Press,Psychology Press,English,English,False,0,Hardcover,Tapa dura,True
7,183242,177086,Detik-Detik Diri Di Daerah Daif,191,Lecturas Ágiles,4.38,"4,0 a 4,49",77,17,0.2208,"Dewan Bahasa dan Pustaka, Kementerian Pendidik...","Dewan Bahasa dan Pustaka, Kementerian Pendidik...",Malay,Malay,False,0,Paperback,Tapa blanda,True
11,2501455,2508820,فوضى الحواس,375,Obras extensas,3.78,"3,5 a 3,99",20120,1462,0.0727,دار الآداب,دار الآداب,Arabic,Arabic,True,1,Paperback,Tapa blanda,False
13,985576,971071,"Next door, down the road, around the corner;: ...",188,Lecturas Ágiles,5.00,"4,5 a 5,0",1,0,0.0000,Doubleday,Doubleday,English,English,False,0,Paperback,Tapa blanda,True


Comprobación de las reglas de normalización: etiquetas de idioma que cambian y reparto de los formatos agrupados.

In [6]:
cambios_idioma = (
    df_final.loc[df_final["language_original"].ne(df_final["idioma_normalizado"])
                 & df_final["language_original"].notna()]
    .groupby(["language_original", "idioma_normalizado"]).size()
    .rename("registros").sort_values(ascending=False).reset_index()
)
grupos_formato = df_final["formato_grupo"].value_counts().rename("registros").to_frame()
grupos_formato["porcentaje"] = (grupos_formato["registros"] / len(df_final) * 100).round(2)
display(cambios_idioma)
grupos_formato

,language_original,idioma_normalizado,registros
0,Spanish; Castilian,Spanish,7550
1,Dutch; Flemish,Dutch,998
2,Catalan; Valencian,Catalan,87
3,"Bokmål, Norwegian; Norwegian Bokmål","Bokmål, Norwegian",22
4,Filipino; Pilipino,Filipino,19
5,Sinhala; Sinhalese,Sinhala,8
6,"Norwegian Nynorsk; Nynorsk, Norwegian",Norwegian Nynorsk,5
7,Gaelic; Scottish Gaelic,Gaelic,2
8,Luxembourgish; Letzeburgesch,Luxembourgish,1
9,Klingon; tlhIngan-Hol,Klingon,1


,registros,porcentaje
formato_grupo,,
Tapa blanda,133716,56.88
Tapa dura,80970,34.44
Digital,12860,5.47
Sin informar,3746,1.59
Audio,2259,0.96
Otro,1553,0.66


## 5. Respuestas a las preguntas de negocio

Convenciones de todas las tablas: un **registro** es una edición (`book_id`); una **obra** es un `work_id`. Las calificaciones están en una escala de 1 a 5 y las páginas en número de páginas. Los porcentajes se calculan sobre el total indicado en cada tabla.

### P1. Población de estudio y exclusiones

In [7]:
t1_control_poblacion

,criterio,registros,porcentaje_del_dataset
0,Dataset original,500000,100.00
1,Calificación faltante,258,0.05
2,Calificación < 3.5,223590,44.72
3,Calificación >= 3.5 (destacados),276152,55.23
4,Destacados con páginas faltantes,22129,4.43
5,Destacados con páginas < 1,18868,3.77
6,Destacados con páginas > 5.000,51,0.01
7,Población final: destacados con páginas entre ...,235104,47.02


**Respuesta P1.** De los 500.000 registros, 276.152 (55,23 %) tienen calificación promedio ≥ 3,5 y forman la población destacada. Los 223.590 registros con calificación menor a 3,5 incluyen 148.219 obras sin ninguna calificación (`average_rating = 0` y `ratings_count = 0`): ese 0 indica ausencia de valoraciones, no una mala nota. Dentro de los destacados se excluyen 41.048 registros (14,86 %) que no permiten asignar un área: 22.129 sin páginas, 18.868 con 0 páginas y 51 con más de 5.000. La población final es de **235.104 registros** (85,14 % de los destacados; 47,02 % del dataset).

*Utilidad:* la librería sabe exactamente sobre qué libros se calcula cada resultado. *Limitación:* el rango de 1 a 5.000 páginas solo elimina errores evidentes; dentro de él puede haber valores dudosos, como folletos de muy pocas páginas o audiolibros que registran minutos.

### P2. Reparto por extensión

In [8]:
asignacion_obra = df_final.loc[df_final["registro_representativo_obra"]]
t2_reparto_por_extension = (
    df_final.groupby("categoria_extension")
    .agg(
        registros_edicion=("book_id", "size"),
        obras_con_ediciones_en_area=("work_id", "nunique"),
        paginas_min=("pages", "min"),
        paginas_mediana=("pages", "median"),
        paginas_max=("pages", "max"),
    )
    .reindex(prep.ORDEN_AREAS)
)
t2_reparto_por_extension.insert(
    1, "porcentaje_registros",
    (t2_reparto_por_extension["registros_edicion"] / len(df_final) * 100).round(2),
)
t2_reparto_por_extension.insert(
    3, "obras_asignadas", asignacion_obra["categoria_extension"].value_counts().reindex(prep.ORDEN_AREAS)
)
t2_reparto_por_extension.insert(
    4, "porcentaje_obras",
    (t2_reparto_por_extension["obras_asignadas"] / len(asignacion_obra) * 100).round(2),
)
t2_reparto_por_extension = t2_reparto_por_extension.reset_index()

assert t2_reparto_por_extension["registros_edicion"].sum() == len(df_final)
assert t2_reparto_por_extension["obras_asignadas"].sum() == df_final["work_id"].nunique()
assert abs(t2_reparto_por_extension["porcentaje_registros"].sum() - 100) < 0.01
t2_reparto_por_extension

,categoria_extension,registros_edicion,porcentaje_registros,obras_con_ediciones_en_area,obras_asignadas,porcentaje_obras,paginas_min,paginas_mediana,paginas_max
0,Lecturas Ágiles,175788,74.77,162287,161020,75.96,1,192.0,350
1,Obras extensas,59316,25.23,52194,50955,24.04,351,453.0,4991


In [9]:
percentiles_paginas = df_final["pages"].astype(float).quantile([0.25, 0.5, 0.75]).rename("paginas")
t2b_sensibilidad_umbral = prep.sensibilidad_umbral(df_final, UMBRALES_CANDIDATOS)
display(percentiles_paginas.to_frame())
t2b_sensibilidad_umbral

,paginas
0.25,144.0
0.50,240.0
0.75,352.0


,umbral_paginas,categoria_extension,registros_edicion,porcentaje_registros,obras_unicas,obras_con_ediciones_en_ambas_areas
0,250,Lecturas Ágiles,123187,52.40,115082,2781
1,250,Obras extensas,111917,47.60,99674,2781
2,300,Lecturas Ágiles,152779,64.98,141809,2662
3,300,Obras extensas,82325,35.02,72828,2662
4,350,Lecturas Ágiles,175788,74.77,162287,2506
5,350,Obras extensas,59316,25.23,52194,2506
6,400,Lecturas Ágiles,194193,82.60,178292,2218
7,400,Obras extensas,40911,17.40,35901,2218


**Respuesta P2.** Con el umbral de 350 páginas, **Lecturas Ágiles reúne 175.788 registros (74,77 %) y Obras extensas 59.316 (25,23 %)**. Contando cada obra una sola vez, el reparto es casi igual: 161.020 obras (75,96 %) frente a 50.955 (24,04 %). La mediana de páginas es de 192 en Lecturas Ágiles y de 453 en Obras extensas.

*Justificación del umbral:* 350 páginas está muy cerca del percentil 75 de la población (352), así que separa el cuarto de libros más extensos. La tabla de sensibilidad muestra que el reparto depende mucho del corte: con 250 páginas sería 52,4 % / 47,6 % y con 400 páginas, 82,6 % / 17,4 %. Solo 2.506 obras (1,18 %) tienen ediciones a ambos lados del umbral, por lo que la asignación por obra es estable.

*Utilidad:* como referencia inicial, unas tres cuartas partes de los títulos destacados corresponden a lecturas ágiles. *Limitación:* la proporción de títulos no equivale a la proporción de espacio físico; un libro extenso ocupa más lomo, y la decisión final depende de las dimensiones reales de los estantes.

### P3. Editoriales con más títulos destacados por área

In [10]:
t3_top_editoriales = prep.crear_ranking(df_final, "publisher")
t3_top_editoriales

,categoria_extension,posicion,publisher,registros_edicion,porcentaje_registros_con_valor,obras_unicas,calificacion_promedio,calificacion_mediana
0,Lecturas Ágiles,1,Routledge,2296,1.34,2246,4.09,4.00
1,Lecturas Ágiles,2,Oxford University Press,1499,0.88,1457,4.06,4.00
2,Lecturas Ágiles,3,Cambridge University Press,1366,0.80,1312,4.06,4.00
3,Lecturas Ágiles,4,HarperCollins,819,0.48,791,3.95,3.94
4,Lecturas Ágiles,5,iUniverse,628,0.37,617,4.35,4.26
5,Lecturas Ágiles,6,Scholastic,617,0.36,614,3.96,3.93
6,Lecturas Ágiles,7,Springer,597,0.35,589,4.26,4.00
7,Lecturas Ágiles,8,Wiley,517,0.30,508,4.03,4.00
8,Lecturas Ágiles,9,Kessinger Publishing,514,0.30,510,4.08,4.00
9,Lecturas Ágiles,10,Palgrave Macmillan,510,0.30,502,4.09,4.00


**Respuesta P3.** En ambas áreas predominan las editoriales académicas. En Lecturas Ágiles encabezan Routledge (2.296 registros; 1,34 %), Oxford University Press (1.499) y Cambridge University Press (1.366); en Obras extensas, Oxford University Press (1.146; 2,00 %), Routledge (791), Cambridge (784), Springer (632) y Wiley (581). Las editoriales comerciales aparecen más abajo: HarperCollins y Scholastic en Lecturas Ágiles, y Bantam y Thorndike Press (letra grande) en Obras extensas. Las calificaciones promedio más altas del top 10 son las de iUniverse (4,35, autopublicación), CRC Press (4,29) y Springer (4,26 y 4,30).

*Utilidad:* el catálogo está muy fragmentado (47.002 editoriales; la primera no llega al 2 % de su área), así que negociar con pocas editoriales cubre una parte pequeña de los títulos destacados. *Limitación:* los nombres no se unificaron entre sellos de un mismo grupo (p. ej., Penguin Books y Penguin Classics), y la fuerte presencia académica refleja el catálogo de Goodreads, no la demanda de una librería general.

### P4. Idiomas con más títulos destacados por área

In [11]:
t4_top_idiomas = prep.crear_ranking(df_final, "idioma_normalizado")
t4_top_idiomas

,categoria_extension,posicion,idioma_normalizado,registros_edicion,porcentaje_registros_con_valor,obras_unicas,calificacion_promedio,calificacion_mediana
0,Lecturas Ágiles,1,English,147959,86.26,139094,4.07,4.00
1,Lecturas Ágiles,2,Spanish,5647,3.29,5412,4.04,4.00
2,Lecturas Ágiles,3,French,4740,2.76,4587,4.00,3.97
3,Lecturas Ágiles,4,German,3835,2.24,3712,4.03,4.00
4,Lecturas Ágiles,5,Italian,1332,0.78,1308,3.95,3.90
5,Lecturas Ágiles,6,Japanese,1206,0.70,1200,4.12,4.02
6,Lecturas Ágiles,7,Indonesian,1058,0.62,1055,4.01,4.00
7,Lecturas Ágiles,8,Portuguese,872,0.51,852,3.99,3.95
8,Lecturas Ágiles,9,Dutch,701,0.41,696,3.92,3.87
9,Lecturas Ágiles,10,Arabic,603,0.35,598,3.90,3.86


**Respuesta P4.** El inglés concentra el **86,26 %** de los registros con idioma informado en Lecturas Ágiles y el **83,32 %** en Obras extensas. En Lecturas Ágiles le siguen el español (5.647; 3,29 %), el francés (2,76 %) y el alemán (2,24 %). En Obras extensas el orden cambia: alemán (3,65 %), francés (3,47 %) y español (1.903; 3,28 %). Las calificaciones promedio de los diez idiomas van de 3,90 a 4,12; el japonés tiene la más alta en ambas áreas (4,12) y el español obtiene 4,04 y 4,07.

La normalización unificó 7.550 registros con la etiqueta "Spanish; Castilian" en "Spanish" y 998 con "Dutch; Flemish" en "Dutch", sin cambiar el número de registros.

*Utilidad:* para compras importadas, el inglés es el idioma con más oferta destacada en las dos áreas; alemán y francés pesan algo más entre las obras extensas. *Limitación:* Goodreads tiene más usuarios angloparlantes, por lo que la proporción por idioma no representa el mercado local.

### P5. Distribución de la calificación promedio por área

In [12]:
estadisticas_calificacion = (
    df_final.groupby("categoria_extension")["average_rating"]
    .describe()
    .rename(columns={
        "count": "registros_edicion", "mean": "calificacion_media", "std": "desviacion_estandar",
        "min": "calificacion_min", "25%": "calificacion_p25", "50%": "calificacion_mediana",
        "75%": "calificacion_p75", "max": "calificacion_max",
    })
)
porcentaje_por_rango = (
    pd.crosstab(df_final["categoria_extension"], df_final["rango_calificacion"], normalize="index")
    .reindex(columns=prep.ORDEN_RANGOS_CALIFICACION) * 100
).add_prefix("porcentaje_rango_")
t5_calificacion_por_extension = (
    estadisticas_calificacion.join(porcentaje_por_rango).reindex(prep.ORDEN_AREAS).round(2).reset_index()
)
t5_calificacion_por_extension["registros_edicion"] = t5_calificacion_por_extension["registros_edicion"].astype(int)

columnas_rango = [c for c in t5_calificacion_por_extension if c.startswith("porcentaje_rango_")]
assert (t5_calificacion_por_extension[columnas_rango].sum(axis=1) - 100).abs().max() < 0.05
t5_calificacion_por_extension

,categoria_extension,registros_edicion,calificacion_media,desviacion_estandar,calificacion_min,calificacion_p25,calificacion_mediana,calificacion_p75,calificacion_max,"porcentaje_rango_3,5 a 3,99","porcentaje_rango_4,0 a 4,49","porcentaje_rango_4,5 a 5,0"
0,Lecturas Ágiles,175788,4.07,0.40,3.5,3.78,4.0,4.25,5.0,44.20,40.10,15.70
1,Obras extensas,59316,4.08,0.37,3.5,3.82,4.0,4.25,5.0,42.12,44.26,13.62


**Respuesta P5.** La calificación prácticamente no cambia entre áreas: media de 4,07 en Lecturas Ágiles y 4,08 en Obras extensas, la misma mediana (4,00) y el mismo percentil 75 (4,25). Obras extensas es algo más homogénea (desviación 0,37 frente a 0,40; percentil 25 de 3,82 frente a 3,78) y concentra más registros en el rango de 4,0 a 4,49 (44,26 % frente a 40,10 %). Lecturas Ágiles tiene más casos en los extremos: 44,20 % entre 3,5 y 3,99 y 15,70 % entre 4,5 y 5,0 (frente a 42,12 % y 13,62 %).

*Utilidad:* la calificación no distingue a un área de la otra, así que el reparto del espacio debe basarse en la cantidad de títulos y en el catálogo, no en la valoración. *Limitación:* la población está truncada en 3,5 y `average_rating` es el promedio de la obra, compartido por sus ediciones.

### P6. Composición por idioma de cada área

In [13]:
IDIOMAS_DESTACADOS = 5
idioma_area = df_final["idioma_normalizado"].fillna("Sin idioma informado")
principales = df_final["idioma_normalizado"].value_counts().head(IDIOMAS_DESTACADOS).index
idioma_area = idioma_area.where(
    idioma_area.isin(principales) | idioma_area.eq("Sin idioma informado"), "Otros idiomas"
)
orden_idiomas = [*principales, "Otros idiomas", "Sin idioma informado"]

registros = pd.crosstab(idioma_area, df_final["categoria_extension"]).reindex(
    index=orden_idiomas, columns=prep.ORDEN_AREAS
)
porcentajes = (registros / registros.sum() * 100).round(2)
t6_composicion_idioma = (
    registros.add_prefix("registros_").join(porcentajes.add_prefix("porcentaje_"))
    .rename_axis("idioma").reset_index()
)
assert (porcentajes.sum() - 100).abs().max() < 0.05
assert registros.to_numpy().sum() == len(df_final)
t6_composicion_idioma

categoria_extension,idioma,registros_Lecturas Ágiles,registros_Obras extensas,porcentaje_Lecturas Ágiles,porcentaje_Obras extensas
0,English,147959,48334,84.17,81.49
1,Spanish,5647,1903,3.21,3.21
2,French,4740,2011,2.70,3.39
3,German,3835,2115,2.18,3.57
4,Italian,1332,584,0.76,0.98
5,Otros idiomas,8006,3062,4.55,5.16
6,Sin idioma informado,4269,1307,2.43,2.20


**Respuesta P6.** Sobre el total de registros de cada área (incluidos los que no tienen idioma), el inglés representa el 84,17 % de Lecturas Ágiles y el 81,49 % de Obras extensas. El español tiene la misma participación en ambas (3,21 %). El francés (3,39 %) y el alemán (3,57 %) pesan más en Obras extensas que en Lecturas Ágiles (2,70 % y 2,18 %), y los demás idiomas suman el 4,55 % y el 5,16 %. No tienen idioma informado el 2,43 % y el 2,20 %.

*Utilidad:* Obras extensas es un poco más diversa en idiomas europeos, lo que sugiere reservar en esa área un espacio para importaciones en alemán y francés. *Limitación:* con solo un 3,21 % de registros en español, cualquier conclusión para una librería hispanohablante debe tomarse con cautela.

### Pregunta complementaria (ajuste). Participación de lectores, series y formato por área

`ratings_count` y `reviews_count` se agregan a nivel de obra en la fuente, por eso las sumas usan solo el registro representativo de cada obra (`registro_representativo_obra`); así cada obra cuenta una vez y queda en el área de su edición con más calificaciones. Las proporciones de series se informan por registro y por obra.

In [14]:
obras = df_final.loc[df_final["registro_representativo_obra"]]
t7_participacion_series = obras.groupby("categoria_extension").agg(
    obras_asignadas=("work_id", "size"),
    calificaciones_totales=("ratings_count", "sum"),
    resenas_totales=("reviews_count", "sum"),
    calificaciones_mediana_por_obra=("ratings_count", "median"),
    resenas_mediana_por_obra=("reviews_count", "median"),
    obras_en_series=("is_part_of_series", "sum"),
).reindex(prep.ORDEN_AREAS)
t7_participacion_series.insert(
    2, "porcentaje_calificaciones",
    t7_participacion_series["calificaciones_totales"] / t7_participacion_series["calificaciones_totales"].sum() * 100,
)
t7_participacion_series.insert(
    4, "porcentaje_resenas",
    t7_participacion_series["resenas_totales"] / t7_participacion_series["resenas_totales"].sum() * 100,
)
t7_participacion_series["resenas_por_cada_100_calificaciones"] = (
    t7_participacion_series["resenas_totales"] / t7_participacion_series["calificaciones_totales"] * 100
)
t7_participacion_series["porcentaje_obras_en_series"] = (
    t7_participacion_series["obras_en_series"] / t7_participacion_series["obras_asignadas"] * 100
)
t7_participacion_series["porcentaje_registros_en_series"] = (
    df_final.groupby("categoria_extension")["is_part_of_series"].mean().reindex(prep.ORDEN_AREAS) * 100
)
t7_participacion_series = t7_participacion_series.round(2).reset_index()

assert t7_participacion_series["calificaciones_totales"].sum() == obras["ratings_count"].sum()
assert abs(t7_participacion_series["porcentaje_calificaciones"].sum() - 100) < 0.05
t7_participacion_series

,categoria_extension,obras_asignadas,calificaciones_totales,porcentaje_calificaciones,resenas_totales,porcentaje_resenas,calificaciones_mediana_por_obra,resenas_mediana_por_obra,obras_en_series,resenas_por_cada_100_calificaciones,porcentaje_obras_en_series,porcentaje_registros_en_series
0,Lecturas Ágiles,161020,480178210,56.17,23748290,58.57,13.0,2.0,27622,4.95,17.15,18.37
1,Obras extensas,50955,374649243,43.83,16799301,41.43,31.0,3.0,9471,4.48,18.59,20.58


In [15]:
formato_registros = pd.crosstab(df_final["formato_grupo"], df_final["categoria_extension"]).reindex(
    columns=prep.ORDEN_AREAS
)
formato_registros = formato_registros.loc[formato_registros.sum(axis=1).sort_values(ascending=False).index]
t8_formato_por_extension = (
    formato_registros.add_prefix("registros_")
    .join((formato_registros / formato_registros.sum() * 100).round(2).add_prefix("porcentaje_"))
    .rename_axis("formato_grupo").reset_index()
)
assert formato_registros.to_numpy().sum() == len(df_final)
t8_formato_por_extension

categoria_extension,formato_grupo,registros_Lecturas Ágiles,registros_Obras extensas,porcentaje_Lecturas Ágiles,porcentaje_Obras extensas
0,Tapa blanda,102351,31365,58.22,52.88
1,Tapa dura,58166,22804,33.09,38.44
2,Digital,8858,4002,5.04,6.75
3,Sin informar,2981,765,1.70,1.29
4,Audio,2162,97,1.23,0.16
5,Otro,1270,283,0.72,0.48


**Respuesta complementaria.** Contando cada obra una sola vez, **Obras extensas tiene el 24,04 % de las obras pero reúne el 43,83 % de las calificaciones (374,6 millones) y el 41,43 % de las reseñas (16,8 millones)**. Lecturas Ágiles reúne el 56,17 % de las calificaciones (480,2 millones) y el 58,57 % de las reseñas (23,7 millones). La mediana por obra confirma la diferencia: 31 calificaciones y 3 reseñas en Obras extensas, frente a 13 y 2 en Lecturas Ágiles. La proporción de calificaciones con reseña es similar (4,95 y 4,48 reseñas por cada 100 calificaciones). Las obras en series pesan un poco más en Obras extensas (18,59 % de las obras frente a 17,15 %). En formato, ambas áreas son mayoritariamente de tapa blanda (58,22 % y 52,88 %); la tapa dura (38,44 % frente a 33,09 %) y lo digital (6,75 % frente a 5,04 %) pesan más en Obras extensas, y el audio casi solo aparece en Lecturas Ágiles (1,23 %).

*Utilidad:* aunque hay menos obras extensas, cada una atrae en promedio más lectores. Eso justifica no reducir esa área solo en proporción al número de títulos. *Limitación:* las sumas están dominadas por unas pocas obras muy populares (máximo de 11,9 millones de calificaciones), por eso se informan también las medianas. En el audio, las "páginas" pueden corresponder a otra unidad, y 131 registros tienen más reseñas que calificaciones, una inconsistencia de la fuente que no afecta las sumas por obra.

## 6. Exportación del dataset preparado, las tablas y el resumen descriptivo

Cada ejecución crea `AE3/resultados/` si no existe y sobrescribe los archivos (UTF-8, rutas relativas a la raíz del repositorio). Después se leen de nuevo para verificar que coinciden con los DataFrames del notebook.

In [16]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
exportaciones = {
    "libros_preparados_enriquecidos.csv": df_final,
    "t1_control_poblacion.csv": t1_control_poblacion,
    "t2_reparto_por_extension.csv": t2_reparto_por_extension,
    "t2b_sensibilidad_umbral.csv": t2b_sensibilidad_umbral,
    "t3_top_editoriales.csv": t3_top_editoriales,
    "t4_top_idiomas.csv": t4_top_idiomas,
    "t5_calificacion_por_extension.csv": t5_calificacion_por_extension,
    "t6_composicion_idioma.csv": t6_composicion_idioma,
    "t7_participacion_series.csv": t7_participacion_series,
    "t8_formato_por_extension.csv": t8_formato_por_extension,
}
for nombre, tabla in exportaciones.items():
    tabla.to_csv(RUTA_RESULTADOS / nombre, index=False, encoding="utf-8")

reportes = Reportes()
metadatos = {
    "Dataset": DATASET_ID,
    "Enlace": DATASET_URL,
    "Configuración": CONFIG,
    "Split": SPLIT,
    "Revisión": REVISION,
    "Muestra": "No; se parte del split train completo de la revisión indicada",
    "DataFrame resumido": "df_final (transformado y enriquecido en AE3)",
    "Filtro": f"average_rating >= {CALIFICACION_MINIMA} y páginas entre {PAGINAS_MIN} y {PAGINAS_MAX}",
    "Umbral de extensión": f"{UMBRAL_PAGINAS} páginas (Lecturas Ágiles <= {UMBRAL_PAGINAS} < Obras extensas)",
    "Unidad": "Registro de edición (book_id); work_id identifica la obra",
}
ruta_resumen = reportes.guardar_resumen_dataframe(
    df_final, RUTA_RESULTADOS / "resumen_dataframe_final.txt", metadatos, incluir_estadisticas=True
)

# Verificación: se leen de nuevo los archivos generados
for nombre, tabla in exportaciones.items():
    leida = pd.read_csv(RUTA_RESULTADOS / nombre)
    assert leida.shape == tabla.shape, nombre
    print(f"{(RUTA_RESULTADOS / nombre).relative_to(ROOT).as_posix()}: {leida.shape[0]:,} filas x {leida.shape[1]} columnas")
print(f"{ruta_resumen.relative_to(ROOT).as_posix()}\n")
print(ruta_resumen.read_text(encoding="utf-8"))

AE3/resultados/libros_preparados_enriquecidos.csv: 235,104 filas x 19 columnas
AE3/resultados/t1_control_poblacion.csv: 8 filas x 3 columnas
AE3/resultados/t2_reparto_por_extension.csv: 2 filas x 9 columnas
AE3/resultados/t2b_sensibilidad_umbral.csv: 8 filas x 6 columnas
AE3/resultados/t3_top_editoriales.csv: 20 filas x 8 columnas
AE3/resultados/t4_top_idiomas.csv: 20 filas x 8 columnas
AE3/resultados/t5_calificacion_por_extension.csv: 2 filas x 12 columnas
AE3/resultados/t6_composicion_idioma.csv: 7 filas x 5 columnas
AE3/resultados/t7_participacion_series.csv: 2 filas x 12 columnas
AE3/resultados/t8_formato_por_extension.csv: 6 filas x 5 columnas
AE3/resultados/resumen_dataframe_final.txt

RESUMEN DEL DATAFRAME
Dataset: pfaha/goodreads-books
Enlace: https://huggingface.co/datasets/pfaha/goodreads-books
Configuración: default
Split: train
Revisión: 43b3198cd7ec9c142d56181f1a09ed9e8c112610
Muestra: No; se parte del split train completo de la revisión indicada
DataFrame resumido: df_fin

## 7. Conclusiones del avance

1. **Reparto:** con el umbral de 350 páginas, alrededor de **tres de cada cuatro títulos destacados son lecturas ágiles** (74,77 % de los registros; 75,96 % de las obras). Este reparto depende del umbral, que sigue siendo provisional.
2. **Calificación:** no diferencia las áreas (4,07 frente a 4,08), por lo que no sirve como criterio para repartir el espacio.
3. **Participación:** las obras extensas son menos, pero concentran una parte desproporcionada de las calificaciones (43,83 %) y de las reseñas (41,43 %). Es un argumento para darles más espacio que el que indica solo su número de títulos.
4. **Catálogo:** está dominado por el inglés y por editoriales académicas, y es muy fragmentado. Las compras importadas pueden priorizar el inglés y, en obras extensas, alemán y francés.

**Pendiente para las siguientes etapas:**
- validar el umbral con las dimensiones reales de los estantes;
- unificar editoriales que pertenecen a un mismo grupo;
- contrastar con datos de ventas e inventario, que el dataset no tiene.

Las tablas exportadas en `AE3/resultados/` quedan listas para las visualizaciones de AE4.